# 30 — Agentic RAG

Retrieval as **one tool among many** in a ReAct loop — the agent chooses per turn whether to search the internal KB, hit the web, or run the calculator.

**What you'll learn**
- `create_react_agent` from `langgraph.prebuilt` — the simplest way to wire a tool-calling agent
- The `@tool` decorator: docstring = tool description the agent reads to decide when to call it
- Three tools: `vectorstore_search` (private KB), `web_search` (DuckDuckGo), `calculator` (`eval` sandboxed)
- Why agentic RAG is more flexible than pipeline RAG: the agent decides per turn, not a fixed flow
- The safety pattern for `eval()` — `{'__builtins__': {}}` blocks arbitrary code execution

**Companion examples:** 4-basic-react-agent (ReAct basics), 25-adaptive-rag (rule-based routing), 11-hitl-approval (pause agent for human review)

In [ ]:
# ── 1. Install dependencies (Colab only) ──────────────────────────────────────
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai langchain-chroma langchain-community chromadb duckduckgo-search python-dotenv langgraph

In [ ]:
# ── 2. API key ─────────────────────────────────────────────────────────────────
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()

if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. Define the three tools ─────────────────────────────────────────────────
# @tool converts any function into a tool the agent can call.
# The DOCSTRING is what the agent reads to decide when to use each tool.
from langchain_chroma import Chroma
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_openai import OpenAIEmbeddings

_DOCS = [
    Document(page_content="LangGraph is a library for building stateful, multi-actor applications with LLMs."),
    Document(page_content="RAG combines a retriever with a generative model to ground answers in source documents."),
    Document(page_content="Self-RAG uses reflection tokens to decide whether retrieval is needed per query."),
    Document(page_content="Human-in-the-loop checkpointing pauses graph execution and waits for human approval."),
    Document(page_content="Adaptive RAG classifies each query and routes it to the cheapest correct retrieval path."),
]

_store = Chroma.from_documents(_DOCS, OpenAIEmbeddings())
_retriever = _store.as_retriever(search_kwargs={"k": 2})
_duck = DuckDuckGoSearchResults(max_results=3)


@tool
def vectorstore_search(query: str) -> str:
    """Search the internal knowledge base about LangGraph and RAG patterns."""
    docs = _retriever.invoke(query)
    return "\n\n".join(d.page_content for d in docs) or "No results found."


@tool
def web_search(query: str) -> str:
    """Search the web using DuckDuckGo for current information."""
    return _duck.run(query)


@tool
def calculator(expression: str) -> str:
    """Evaluate a Python math expression. Example: '2 ** 10' or '(5 + 3) * 7'."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))  # noqa: S307
    except Exception as e:
        return f"Error: {e}"


# Quick sanity check before wiring into the agent
print("vectorstore:", vectorstore_search.invoke({"query": "What is LangGraph?"})[:80])
print("calculator :", calculator.invoke({"expression": "144 / 12 * 7"}))

In [ ]:
# ── 4. Build the agentic RAG agent ────────────────────────────────────────────
# create_react_agent wires LLM + tools into a full ReAct loop automatically.
# No manual nodes needed — it handles tool-call → observe → continue.
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

SYSTEM = (
    "You are a research assistant with three tools:\n"
    "- vectorstore_search: internal knowledge base about LangGraph and RAG\n"
    "- web_search: live web results via DuckDuckGo\n"
    "- calculator: Python math expressions\n"
    "For internal docs questions use vectorstore_search first. "
    "For math use calculator. For current events use web_search."
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent = create_react_agent(llm, tools=[vectorstore_search, web_search, calculator], prompt=SYSTEM)

print("Agent ready — tools: vectorstore_search, web_search, calculator")

In [ ]:
# ── 5. Run the agent on mixed questions ───────────────────────────────────────
from langchain_core.messages import HumanMessage

QUESTIONS = [
    "What is Self-RAG and how does it differ from Corrective RAG?",  # → vectorstore
    "What is 144 divided by 12, then multiplied by 7?",             # → calculator
    "Who won the most recent FIFA World Cup?",                       # → web search
]

for q in QUESTIONS:
    print(f"\n{'─'*60}")
    print(f"Q: {q}")
    result = agent.invoke({"messages": [HumanMessage(content=q)]})
    print(f"A: {result['messages'][-1].content[:300]}")

In [ ]:
# ── 6. Inspect the full ReAct trace ───────────────────────────────────────────
# Run one question and print every message to see the tool-call loop in detail.
result = agent.invoke({"messages": [HumanMessage(content="What is Adaptive RAG?")]})

for msg in result["messages"]:
    role = type(msg).__name__
    content = str(msg.content)[:120] if msg.content else ""
    print(f"[{role}] {content}")

## Exercises

**Exercise 1 — Add a fourth tool:** Create a `@tool` function `get_date()` that returns today's date as a string. Add it to the agent and ask `"What day of the week is today?"` — does the agent use it?

**Exercise 2 — Remove the system prompt hints:** Delete the tool selection hints from `SYSTEM`. Ask the same three questions. Does the agent still route correctly without guidance?

**Exercise 3 — Multi-step question:** Ask `"Compare the API rate limits for Pro and Free plans, then tell me how many Free requests fit into one Pro request"`. This requires both vectorstore (policy) and calculator. Watch the trace in cell 6.

## What's next

- **4-basic-react-agent** — ReAct pattern basics with simpler tool set
- **25-adaptive-rag** — rule-based routing vs. agent-driven tool choice
- **11-hitl-approval** — pause the agent mid-run and require human sign-off before continuing